## Downloading titles directly from the release

The figures on this page are computed from a handful of CSVs that
[`aggregate_titles.py`](https://github.com/INGEOTEC/LegalIA/blob/master/website/scripts/aggregate_titles.py)
pre-aggregates from a full local checkout of the `notas-archivo` release (see
[Dataset](../dataset.ipynb)). `dofjson.titulos` offers a lighter,
Colab-friendly alternative: pulling titles straight out of the release, with
no local checkout at all.

`dofjson.titulos.download_titulos` downloads every release asset into
memory, extracts it, and writes a compact `codNota` + `titulo` (title) + `fecha`
(date) dataset — one line per note — to a gzip-compressed JSONL file:

```python
from pathlib import Path
from dofjson.titulos import download_titulos

download_titulos(Path("titulos.jsonl.gz"))
```

`codNota` and `fecha` round out each record — `codNota` to fetch a note's
full content later, `fecha` to place it in time — but neither is needed
for what follows: the release is already split into one asset per year, so
this section reads the year straight off the asset name and keeps only
`titulo`.

## Grouping titles by year

Each note carries its own `fecha`, but there is no need to parse it: every
release asset already covers exactly one year (`notas-YYYY.tgz`) or one
month of the year still in progress (`notas-YYYY-MM.tgz`), so the asset's
own name is enough to know which year every title inside it belongs to.
The loop below walks each asset, keeps only `titulo` — dropping `codNota`
and `fecha` alike — and appends it to that asset's year.

In [1]:
import io
import json
import re
import tarfile
from collections import defaultdict

import requests

from dofjson.titulos import listar_assets

_LISTAS_NOTAS = ("NotasMatutinas", "NotasVespertinas", "NotasExtraordinarias")
_ASSET_YEAR = re.compile(r"notas-(\d{4})(?:-\d{2})?\.tgz")


def titulos_del_asset(asset):
    """Every title inside one notas-YYYY[-MM].tgz asset, plus its year."""
    year = int(_ASSET_YEAR.match(asset["name"]).group(1))
    response = requests.get(asset["url"], timeout=60)
    response.raise_for_status()
    titulos = []
    with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:gz") as tar:
        for member in tar:
            if not member.isfile() or not member.name.endswith(".json"):
                continue
            dia = json.load(tar.extractfile(member))
            for lista in _LISTAS_NOTAS:
                for nota in dia.get(lista, []):
                    if nota.get("titulo"):
                        titulos.append(nota["titulo"])
    return year, titulos

In [2]:
assets = listar_assets()
titles_by_year = defaultdict(list)

for i, asset in enumerate(assets, 1):
    year, titulos = titulos_del_asset(asset)
    titles_by_year[year].extend(titulos)
    print(f"[{i}/{len(assets)}] {asset['name']}: {len(titulos)} titles -> {year}")

titles_by_year = sorted(titles_by_year.items())

[1/115] notas-1917.tgz: 2670 titles -> 1917


[2/115] notas-1918.tgz: 2999 titles -> 1918


[3/115] notas-1919.tgz: 3244 titles -> 1919


[4/115] notas-1920.tgz: 3429 titles -> 1920


[5/115] notas-1921.tgz: 4367 titles -> 1921


...

[111/115] notas-2026-02.tgz: 1532 titles -> 2026


[112/115] notas-2026-03.tgz: 2069 titles -> 2026


[113/115] notas-2026-04.tgz: 2135 titles -> 2026


[114/115] notas-2026-05.tgz: 2065 titles -> 2026


[115/115] notas-2026-06.tgz: 2498 titles -> 2026


`titles_by_year` is now a list of `(year, titles)` pairs, sorted by year,
merging every monthly asset of the year still in progress into a single
entry — the same shape `aggregate_titles.py` builds from a local checkout,
ready to feed this page's per-year figures without one.

In [3]:
years = [year for year, _ in titles_by_year]
total = sum(len(titulos) for _, titulos in titles_by_year)
print(f"{len(titles_by_year)} years, {years[0]}-{years[-1]}, {total:,} titles total")
titles_by_year[0][0], titles_by_year[0][1][:2]

110 years, 1917-2026, 1,232,802 titles total


(1917,
 ['CIRCULAR nº. 164, relativa a los asuntos que se traten por telégrafo con esta Secretaría',
  'SOLICITUD presentada ante esta Secretaría por el Sr. Pastor Hernández pidiendo se le otorgue concesión para aprovechar en riego la cantidad de 216 litros por segundo de las aguas del "Arroyo Hondo", ubicado en el E. de Querétaro'])